In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import torch
import psutil
import sys
import os
from matplotlib.ticker import ScalarFormatter

In [2]:
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.default'] = 'rm'

plt.rc("font", family="serif", size=30)
plt.rc("axes", titlesize="medium")

plt.rcParams['xtick.labelsize'] = 30
plt.rcParams['ytick.labelsize'] = 30

plt.rcParams["axes.formatter.limits"] = [-3,3]

rect_double    = (0.05, 0.12, 0.98, 0.97) # left, bottom, right, top
rect_double_with_legend = (0.14, 0.12, 0.98, 0.97) # left, bottom, right, top

In [3]:
cd C:\Users\flori\OneDrive\Daten\Promotion\Machine Learning\CaloINN\src

C:\Users\flori\OneDrive\Daten\Promotion\Machine Learning\CaloINN\src


c:\Users\flori\anaconda3\envs\CaloINN\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
from myDataLoader import MyDataLoader
import data_util

In [5]:
print(psutil.virtual_memory())
torch.set_default_dtype(torch.float32)

svmem(total=17005826048, available=7088971776, percent=58.3, used=9916854272, free=7088971776)


In [6]:
def load_data(filename, used_layers=None, ):
    """Loads the data for the ML training from an hdf5 file"""
    
    # layer_boundaries = [0, size_layer_0, size_layer_0+size_layer_1, ...]
    # data = {"incident_energy": energy, "energy_layer_0": layer_0, ...}
    # filename = "/afs/cern.ch/user/f/fernst/FrozenShowerInputSamples/eta_020_binned/dataset_eta_020.hdf5"
    
    file = h5py.File(filename, 'r')
    
    if used_layers is None:
        used_layers = [int(key.split("_")[-1]) for key in file.keys() if "bin" not in key and "layer" in key]
        
    used_layers = torch.tensor(used_layers, dtype=torch.int32)    
    used_layers = torch.sort(used_layers)[0]
        
    print(f"Using layers {used_layers}")
        
    # Store the layer energies
    layers = []
    for layer_index in used_layers:
        layers.append(torch.tensor(file[f"energy_layer_{layer_index}"][:], dtype=torch.get_default_dtype()))
    
    # Store the incident energy
    energy = torch.tensor(file["incident_energy"][:], dtype=torch.get_default_dtype())[:, None]
    
    # Create the layer boundaries list
    layer_boundaries = [0]
    for layer in layers:
        layer_boundaries.append(layer.shape[1] + layer_boundaries[-1])
        
    coordinates = [[],[]]
    
    for layer_index in used_layers:
        alpha = torch.tensor(file[f"binsize_alpha_layer_{layer_index}"][:], dtype=torch.get_default_dtype())/2 \
            +   torch.tensor(file[f"binstart_alpha_layer_{layer_index}"][:], dtype=torch.get_default_dtype())
            
        radius = torch.tensor(file[f"binsize_radius_layer_{layer_index}"][:], dtype=torch.get_default_dtype())/2 \
            +    torch.tensor(file[f"binstart_radius_layer_{layer_index}"][:], dtype=torch.get_default_dtype())
        
        eta = radius * torch.cos(alpha)
        phi = radius * torch.sin(alpha)
        
        coordinates[0].append(eta)
        coordinates[1].append(phi)
        
    coordinates[0] = torch.cat(coordinates[0])
    coordinates[1] = torch.cat(coordinates[1])
    
    coordinates = torch.stack(coordinates)
    
    # Concatenate the layers
    x = torch.cat(layers, axis=1)
    
    # Turn x into MeV scale
    x *= energy    

    file.close()
    
    return x, energy, layer_boundaries, coordinates

def separate_negative_energies(x, layer_boundaries):
    negative_layers = []
        
    new_layers = [x]
    for i, (layer_start, layer_end) in enumerate(zip(layer_boundaries[:-1], layer_boundaries[1:])):
        layer = x[..., layer_start:layer_end]
        if torch.any(layer < 0):
            
            negative_layers.append(i)
            
            new_layers.append(-torch.clip(layer, None, 0))
            
            x[..., layer_start:layer_end] = torch.clip(layer, 0, None)
            
            layer_boundaries.append(layer_boundaries[-1] + layer.shape[1])
            
    print(f"Fixed {len(negative_layers)} negative layers")
    
    x = torch.cat(new_layers, axis=1)
    
    return x, layer_boundaries, negative_layers
    
def get_energy_dims(x, c, layer_boundaries, eps=1.e-10):
    """Appends the extra dimensions and the layer energies to the conditions
    The layer energies will always be the last #layers entries, the extra dims will
    be the #layers entries directly after the first entry - the incident energy.
    Inbetween additional features might be appended as further conditions"""
    
    x = torch.clone(x)
    c = torch.clone(c)

    layer_energies = []

    for layer_start, layer_end in zip(layer_boundaries[:-1], layer_boundaries[1:]):
        
        # Compute total energy of current layer
        layer_energy = torch.sum(x[..., layer_start:layer_end], axis=1, keepdims=True)
        
        # Store its energy for later
        layer_energies.append(layer_energy)
        
        
    layer_energies_torch = torch.cat(layer_energies, axis=1)
        
    # Compute the generalized extra dimensions
    extra_dims = [torch.sum(layer_energies_torch, axis=1, keepdims=True) / c]

    for layer_index in range(len(layer_boundaries)-2):
        extra_dim = layer_energies_torch[..., [layer_index]] / (torch.sum(layer_energies_torch[..., layer_index:], axis=1, keepdims=True) + eps)
        extra_dims.append(extra_dim)
        
    # Collect all the conditions
    all_conditions = [c] + extra_dims + layer_energies
    c = torch.cat(all_conditions, axis=1)
    
    return c
            
def preprocess(x, energy, layer_boundaries, eps=1.e-10):
    """Transforms the list 'layers' into the ndarray 'x'. Furthermore, the events
    are masked and the extra dims are appended to the incident energies"""
        
    x, layer_boundaries, negative_layers = separate_negative_energies(x, layer_boundaries)

    binary_mask = torch.full((len(energy),), True)

    # Rescale the energies by an arbitrary factor of 2 -> Only loose O(10) showers instead of ~50%
    # Has to be reversed in the postprocess loop
    x = x/2
    
    # Ensure energy conservation
    binary_mask &= torch.sum(x, axis=1) < energy[:, 0]
    
    # Remove all no-interaction events (= 4 Events in the dataset)
    binary_mask &= torch.sum(x, axis=1) > 0
    
    print(f"Removed {len(energy) - torch.sum(binary_mask)} of {len(energy)} events ({100*(1-torch.sum(binary_mask)/len(energy)):.2f}%)")

    x = x[binary_mask]
    c = energy[binary_mask]

    c = get_energy_dims(x, c, layer_boundaries, eps)
    
    return x, c, negative_layers

def recombine_negative_energies(x, layer_boundaries, negative_layers):
    """Subtracts the negative layers from the positive layers to receive the original data."""
    
    n_layers = len(negative_layers)
    
    for i, layer_index in enumerate(negative_layers):
        positive_layer = x[..., layer_boundaries[layer_index]:layer_boundaries[layer_index+1]]
        negative_layer = x[..., layer_boundaries[-n_layers+i-1]:layer_boundaries[-n_layers+i]]
        
        x[..., layer_boundaries[layer_index]:layer_boundaries[layer_index+1]] = positive_layer - negative_layer
    
    return x[..., :layer_boundaries[-n_layers-1]], layer_boundaries[:-n_layers]

def postprocess(x, c, layer_boundaries, negative_layers, threshold=1e-10, inplace=False):
    """Reverses the effect of the preprocess funtion"""
    
    # Input sanity checks
    assert len(x) == len(c)
    assert len(c.shape) == 2
    assert len(x.shape) == 2
    
    if not inplace:
        # Makes sure, that the original set is not modified inplace
        x = torch.clone(x)
        c = torch.clone(c)
           
    # Set all energies smaller than a threshold to 0. Also prevents negative energies that might occur due to the alpha parameter in
    # the logit preprocessing
    x[x < threshold] = 0.
    
    # Reverse the rescaling that was used before
    x = x*2
    
    x, layer_boundaries = recombine_negative_energies(x, layer_boundaries, negative_layers)
    

    return x, c[..., [0]], layer_boundaries

def get_loaders(filename, val_frac, batch_size, used_layers=None, eps=1.e-10, device='cpu', drop_last=False, shuffle=True, save_memory=False, width_noise=0):
    """Creates the dataloaders used to train the VAE model."""
    
    # load the data from the hdf5 file
    x, energy, layer_boundaries, coordinates = load_data(filename, used_layers=used_layers)

    # preprocess the data and append the extra dims
    x, c, negative_layers = preprocess(x, energy, layer_boundaries, eps)
    
    # Create an index array, used for splitting into train and val set
    number_of_samples = len(x)
    
    # Dont want to mix train and test set, when loading -> No random permutation
    full_index = np.arange(number_of_samples)

    # Split the data
    number_of_val_samples = int(number_of_samples * val_frac)
    number_of_trn_samples = number_of_samples - number_of_val_samples

    trn_index = full_index[:number_of_trn_samples]
    val_index = full_index[number_of_trn_samples:]
    
    if save_memory:
        device = 'cpu'
    
    x_trn = x[trn_index].to(device)
    c_trn = c[trn_index].to(device)
    
    x_val = x[val_index].to(device)
    c_val = c[val_index].to(device)

    # # Call the postprocess func to make sure that it runs through
    # postprocess(x_trn, c_trn, layer_boundaries)
    # postprocess(x_val, c_val, layer_boundaries)
    
    # Create the dataloaders
    trn_loader = MyDataLoader(x_trn, c_trn, batch_size, drop_last, shuffle, width_noise=width_noise)
    val_loader = MyDataLoader(x_val, c_val, batch_size, drop_last, shuffle, width_noise=width_noise)
    return trn_loader, val_loader, layer_boundaries, negative_layers, coordinates
            

In [7]:
filename = r"C:\Users\flori\OneDrive\Daten\Promotion\Machine Learning\Datasets\dataset_eta_020.hdf5"
used_layers = [0,1,2,3,12]
eps = 1.e-10

trn_loader, val_loader, layer_boundaries, negative_layers, coordinates = get_loaders(filename, 0.2, 512, used_layers, eps=eps)

del trn_loader


Using layers tensor([ 0,  1,  2,  3, 12], dtype=torch.int32)
Fixed 1 negative layers
Removed 11 of 127271 events (0.01%)


In [8]:
x, c = val_loader.data, val_loader.cond

In [9]:
x.shape, c.shape, x.element_size() * x.nelement() / 1024**3  # Size in GB

(torch.Size([25452, 6240]), torch.Size([25452, 13]), 0.5916523933410645)

In [10]:
x.max(), x.min(), c.max(), c.min()

(tensor(115457.4375), tensor(-0.), tensor(4194304.), tensor(0.))

In [11]:
x, c = val_loader.data, val_loader.cond
num_detector_layers = len(layer_boundaries) - 1
max_cond_0 = c[:, [0]].max(axis=0, keepdim=True)[0]
max_cond = torch.cat((max_cond_0, torch.ones(1, c.shape[1]-1).to(c.device)), axis=1)*1.15


x_0_1 = data_util.normalize_layers(x, layer_boundaries, c=c, eps=1.e-10)
x_0_1 = x_0_1*0.9          
y_0_1 = torch.cat((x_0_1, (c/max_cond)[:, 0:-num_detector_layers]), axis=1)
assert y_0_1.max() < 1.0, f"y_0_1.max() = {y_0_1.max()}, {torch.argmax(y_0_1)}"
assert y_0_1.min() >= 0.0, f"y_0_1.min() = {y_0_1.min()}, {torch.argmin(y_0_1)}"

In [12]:
x.max(), x.min(), c.max(), c.min()

(tensor(115457.4375), tensor(-0.), tensor(4194304.), tensor(0.))

In [13]:
assert False

AssertionError: 

In [ ]:
x_post, c_post, layer_boundaries_post = postprocess(x, c, layer_boundaries, negative_layers)
print(x_post.shape, c_post.shape)
del x_post, c_post

torch.Size([25452, 4844]) torch.Size([25452, 1])


In [ ]:
def calc_shower_mean(x, c, layer_boundaries, layer, coordinates, direction):
    """Computes the mean of the shower in eta or phi direction for a given layer."""
    
    eta = coordinates[0][layer_boundaries[layer]:layer_boundaries[layer+1]]
    phi = coordinates[1][layer_boundaries[layer]:layer_boundaries[layer+1]]
    
    # Get the layer energy
    layer_energy = x[:, layer_boundaries[layer]:layer_boundaries[layer+1]]
    
    # Compute the mean eta    
    if direction == "eta":
        return np.sum(layer_energy * eta, axis=-1) / (layer_energy.sum(axis=-1)+1.e-16)
    
    elif direction == "phi":
        return np.sum(layer_energy * phi, axis=-1) / (layer_energy.sum(axis=-1)+1.e-16)
    
    else:
        raise ValueError("Invalid direction")
    
def calc_shower_std(x, c, layer_boundaries, layer, coordinates, direction):
    """Computes the std of the shower in eta or phi direction for a given layer."""
    
    mean = calc_shower_mean(x, c, layer_boundaries, layer, coordinates, direction)
    
    eta = coordinates[0][layer_boundaries[layer]:layer_boundaries[layer+1]]
    phi = coordinates[1][layer_boundaries[layer]:layer_boundaries[layer+1]]
    
    # Get the layer energy
    layer_energy = x[:, layer_boundaries[layer]:layer_boundaries[layer+1]]
    
    if direction == "eta":
        discriminant = np.sum(layer_energy * eta * eta, axis=-1) / (layer_energy.sum(axis=-1)+1.e-16) - mean**2
        discriminant[discriminant < 0] = 0
        return np.sqrt(discriminant)
    
    elif direction == "phi":
        discriminant = np.sum(layer_energy * phi * phi, axis=-1) / (layer_energy.sum(axis=-1)+1.e-16) - mean**2
        discriminant[discriminant < 0] = 0
        return np.sqrt(discriminant)
    
    else:
        raise ValueError("Invalid direction")

def calc_flat_energy_distribution(x, c, layer_boundaries, layer=None):
    """Computes the energy distribution of the shower"""
    
    # Factor of two to counter the normalization in the preprocess function
    if layer is None:
        return x.flatten()*2
    
    return x[:, layer_boundaries[layer]:layer_boundaries[layer+1]].flatten()*2
 
def calc_energy(x, c, layer_boundaries, layer=None):
    """Computes the energy of a given layer or of the whole shower"""
    
    # Factor of two to counter the normalization in the preprocess function
    if layer is None:
        return np.sum(x, axis=-1)*2
    
    return np.sum(x[:, layer_boundaries[layer]:layer_boundaries[layer+1]], axis=-1)*2

def calc_etot_over_einc(x, c, layer_boundaries):
    """Computes the total energy of the shower over the incident energy"""
    
    return calc_energy(x, c, layer_boundaries, layer=None) / c
    
    
def get_plot_params(layer_boundaries, coordinates, used_layers=None):
    """Returns the plot parameters for the given layer and direction"""
    
    plots = []
    
    for layer in range(len(layer_boundaries)-1):
        
        if used_layers is not None:
            layer_name = used_layers[layer]
        else:
            layer_name = layer
        
        plots.append(
            [calc_energy, 
             f"energy_{layer_name}.pdf",
             {"layer_boundaries": layer_boundaries, "layer": layer},
             {"axis_label": f'$E_{{\\text{{{layer_name}}}}}$'}]
            )
        
        plots.append(
            [calc_shower_mean, 
             f"mean_{layer_name}_eta.pdf",
             {"layer_boundaries": layer_boundaries, "layer": layer, "coordinates": coordinates, "direction": "eta"},
             {"axis_label": f'$\\langle \\eta \\rangle_{{\\text{{{layer_name}}}}}$'}]
            )
        
        plots.append(
            [calc_shower_std, 
             f"std_{layer_name}_eta.pdf",
             {"layer_boundaries": layer_boundaries, "layer": layer, "coordinates": coordinates, "direction": "eta"},
             {"axis_label": f'$\\sigma_{{\\eta, \\text{{{layer_name}}}}}$'}]
            )
        
        plots.append(
            [calc_shower_mean, 
             f"mean_{layer_name}_phi.pdf",
             {"layer_boundaries": layer_boundaries, "layer": layer, "coordinates": coordinates, "direction": "phi"},
             {"axis_label": f'$\\langle \\phi \\rangle_{{\\text{{{layer_name}}}}}$'}]
            )
        
        plots.append(
            [calc_shower_std, 
             f"std_{layer_name}_phi.pdf",
             {"layer_boundaries": layer_boundaries, "layer": layer, "coordinates": coordinates, "direction": "phi"},
             {"axis_label": f'$\\sigma_{{\\phi, \\text{{{layer_name}}}}}$'}]
            )
        
    plots.append(
        [calc_flat_energy_distribution, 
         "flat_energy_distribution.pdf",
         {"layer_boundaries": layer_boundaries},
         {"axis_label": r'$Voxel distribution$'}]
        )
    
    plots.append(
        [calc_etot_over_einc, 
         "etot_over_einc.pdf",
         {"layer_boundaries": layer_boundaries},
         {"axis_label": r'$E_{tot} / E_{inc}$'}]
        )        

    return plots

def plot_hist(
        file_name,
        data,
        reference,
        axis_label=None,
        xscale='linear',
        yscale='log',
        vmin=None,
        vmax=None,
        n_bins=50,
        ymin=None,
        ymax=None,
        ax=None,
        panel_ax=None,
        panel_scale="linear",
        panel_range=[0.8, 1.2],
        density=True,
        labels=None,
        errorbars_true=False,
        errorbars_fake=False,
        y_label=True, 
        fig = None,
        print_means=False,):
    
    

    if type(errorbars_fake) == bool:
        errorbars_fake = [errorbars_fake]
    
    if type(data)==list and type(data[0])==np.ndarray:
        data_list = data
    else:
        data_list = [data]
             
    if len(errorbars_fake) != len(data_list):
        assert len(errorbars_fake) == 1, "Wrong size for the errorbars index"
        errorbars_fake = [errorbars_fake[0] for _ in range(len(data_list))]
    
    for i in range(len(data_list)):
        finite = np.isfinite(data_list[i])
        data_list[i] = data_list[i][finite]
        
    
    finite = np.isfinite(reference)
    reference = reference[finite]

    all_data = [reference] + data_list

    try:
        # Set the plotting boundaries
        if vmin is None:
            vmin = np.inf
            for elem in all_data:
                vmin = np.min([np.min(elem), vmin])
        if vmax is None:
            vmax = -np.inf
            for elem in all_data:
                vmax = np.max([np.max(elem), vmax])
                
        if vmax == vmin:
            vmax += 0.0001
            vmin -= 0.0001
    except:
        print("Error in setting the boundaries")
        print(axis_label)
        print(all_data)
        return
            
    # Get the bins (Modifications needed if logscale is used)
    if xscale=='log':
        
        if vmin==0:
            vmin = np.inf     
            for elem in all_data:
                vmin = np.min([np.min(elem[elem>1e-7]), vmin])
                
        if isinstance(n_bins, int):
            bins = np.logspace(np.log10(vmin), np.log10(vmax), n_bins)
        else:
            bins = n_bins
    else:
        if isinstance(n_bins, int):
            bins = np.linspace(vmin, vmax, n_bins)
        else:
            bins = n_bins
    

    color = 'blue'
        
    create_fig = False
    if ax is None:
        create_fig = True
        fig, ax = plt.subplots(1,1,figsize=(6,6))    
        
        
    # Plot the reference data
    if not errorbars_true:
        ns_true, bins_true, _ = ax.hist(reference, bins=bins, histtype='stepfilled',
                alpha=0.5, color=color, density=density, label='GEANT', linewidth=1.5)
    else:
        dup_last = lambda a: np.append(a, a[-1])

        bins_true = bins
        
        counts, _ = np.histogram(reference, bins_true, density=False)
        
        ns_true, _ = np.histogram(reference, bins_true, density=density)
        
        mask = (counts == 0)
        counts[mask] = 1
        
        if density: # relative error stays the same
            ref_err = ns_true / np.sqrt(counts)
            
        else:
            ref_err = np.sqrt(ns_true)
        
        
        ref_err[mask] = 0
        
        ax.step(bins_true, dup_last(ns_true), color="blue", alpha=1,
                        linewidth=1.5, where='post', label='GEANT')
        
        ax.step(bins_true, dup_last(ns_true - ref_err), color="blue", alpha=0.5,
                        linewidth=0.5, where='post')
        ax.step(bins_true, dup_last(ns_true + ref_err), color="blue", alpha=0.5,
                        linewidth=0.5, where='post')

        ax.fill_between(bins_true, dup_last(ns_true - ref_err), dup_last(ns_true + ref_err), 
                        facecolor="blue", alpha=0.3, step='post')
    
    # Plot the generated data
    alt_colors = ["green", "red", "orange", "pink", "black"]
    
    ns_fakes = []
    bins_fakes = []
    
    for i, data in enumerate(data_list):
        if not errorbars_fake[i]:
            
            # Modify the labels
            if labels is None:
                label = "VAE"
            else:
                label = labels[i]
            
            ns_i, bins_i, _ = ax.hist(data, bins=bins, histtype='step', linewidth=1.5,
                alpha=1, density=density, label=label, color=alt_colors[i])
    
    
            ns_fakes.append(ns_i)
            bins_fakes.append(bins_i)
            
        else:
            # labels
            if labels is None:
                label = "VAE"
            else:
                label = labels[i]
                
            data = data_list[i]
            
            dup_last = lambda a: np.append(a, a[-1])
            
            bins_i = bins
            
            counts, _ = np.histogram(data, bins_i, density=False)
            
            ns_i, _ = np.histogram(data, bins_i, density=density)
            
            
            mask = (counts == 0)
            counts[mask] = 1
            if density: # relative error stays the same
                data_err = ns_i / np.sqrt(counts)
                
            else:
                data_err = np.sqrt(ns_i)
                
                
            data_err[mask] = 0
            
            ax.step(bins_i, dup_last(ns_i), color=alt_colors[i], alpha=1,
                            linewidth=1.5, where='post', label=label)
            
            ax.step(bins_i, dup_last(ns_i - data_err), color=alt_colors[i], alpha=0.5,
                            linewidth=0.5, where='post')
            ax.step(bins_i, dup_last(ns_i + data_err), color=alt_colors[i], alpha=0.5,
                            linewidth=0.5, where='post')

            ax.fill_between(bins_i, dup_last(ns_i - data_err), dup_last(ns_i + data_err), 
                            facecolor=alt_colors[i], alpha=0.3, step='post')
            
            
            ns_fakes.append(ns_i)
            bins_fakes.append(bins_i)
    
    if y_label:              
        ax.set_ylabel(r"$Normalized counts$")
        
    if panel_ax is not None:
        
        for i in range(len(bins_fakes)):
            assert len(bins_true) == len(bins_fakes[i])
            assert (bins_true - bins_fakes[i] < 1.e-7).all()
        
        for i, (ns_reco, bins_reco) in enumerate(zip(ns_fakes, bins_fakes)):
            
            if labels is not None:
                label = labels[i]
            else:
                label = "VAE"
            
            if i==0:
                mask = ns_true == 0
                ns_true[mask] = 1
                
            panel_data = ns_reco/ns_true
            
            panel_data[mask] = 0
            
            widths = 1.2*(bins_true[1:] - bins_true[:-1])
            panel_ax.axhline(1, color="black", ls="--", alpha=0.5, lw=1
                             )
            panel_ax.hist(bins_reco[:-1], bins_reco[1:]-widths, weights=panel_data, histtype="step", lw=1, ls="--",
                          label=f'{label}/GEANT', color=alt_colors[i])
            
            if y_label:
                panel_ax.set_ylabel(r"$\frac{{Model}}{{GEANT}}$")
        
    ax.set_yscale(yscale)
    ax.set_xscale(xscale)
    if panel_ax is not None:
        panel_ax.set_yscale(panel_scale)
        panel_ax.set_xscale(xscale)

    ax.set_xlim([vmin,vmax])
    if panel_ax is not None:
        panel_ax.set_xlim([vmin,vmax])
        panel_ax.set_ylim(panel_range)
        
    if ymin is not None or ymax is not None:
        ax.set_ylim((ymin, ymax))
        
    if panel_ax is not None:
        lower_bound, upper_bound = ax.get_ylim()
        
        ticks = ax.get_yticks()
        ticks = ticks[ticks >= lower_bound]
        ticks = ticks[ticks <= upper_bound]
        
        if yscale != "log":
            ticks = ticks[1:]

        ax.set_yticks(ticks)
        
        
    if print_means and fig is not None:
        legend = ax.legend(["Data", "Data", "Data"], loc="best")
        plt.draw()
        bbox = legend.get_window_extent().transformed(fig.transFigure.inverted())
        legend.remove()
        
        # print(bbox.x0, bbox.y0)
        
        text = ""
        for i, data in enumerate(all_data):
            mu = np.mean(data)
            std = np.std(data)
            
            if i == 0:
                mu_0 = mu
            
            # text += f'$\mu$={mu:.3e}$\pm${std:.1e}\n'
            # text += f'$\mu$={mu:.3e}\n'
            
            if mu_0 == 0:
                break
            
            text += f'{mu / mu_0:0.3f} \\pm {std / mu_0 / np.sqrt(len(data)):0.3f}\n'
        
        ax.text(bbox.x0, bbox.y0, text, transform=fig.transFigure, fontsize=15)
    

    if axis_label is not None:
        if panel_ax is None:
            ax.set_xlabel(axis_label)
        else:
            panel_ax.set_xlabel(axis_label)
    

    if create_fig:
        fig.tight_layout()
        fig.savefig(file_name, bbox_inches='tight')
        plt.close()
 
def plot_all_hist(xs, cs, plot_params, plot_dir=None, single_plots=False, summary_plot=False, summary_plot_name=None, labels=None, 
                  errorbars_true=False, errorbars_fake=False, plots_per_row=5, plot_seperate_legend=True, ncol=None,
                  print_means=False):

    if plot_dir is not None:
        os.makedirs(plot_dir, exist_ok=True)

    plots = plot_params

    if single_plots and plot_dir is not None:
        
        for i, (func, name, args1, args2) in enumerate(plots):
            
            fig, axs = plt.subplots(2,1, dpi=300, figsize=(7,6*1.3), gridspec_kw={'height_ratios': [1, 0.3]})
            
                            
            # Add ylabels to the leftmost plots
            if i % plots_per_row == 0:
                fig.tight_layout(pad=0.0, w_pad=0.0, h_pad=0.0, rect=rect_double_with_legend)
                ylabel = True
            else:
                fig.tight_layout(pad=0.0, w_pad=0.0, h_pad=0.0, rect=rect_double)
                ylabel = False
                
            plot_hist(
                file_name=None,
                data=[func(x, c, **args1) for x, c in zip(xs[1:], cs[1:])],
                reference=func(xs[0], cs[0], **args1),
                ax=axs[0],
                panel_ax=axs[1],
                labels=labels,
                errorbars_fake=errorbars_fake,
                errorbars_true=errorbars_true,
                y_label=ylabel,
                fig=fig,
                print_means=print_means,
                **args2)

            
            if not plot_seperate_legend:
                # Add figure legend on every rightmost plot
                if i % plots_per_row == plots_per_row-1 or i == len(plots)-1:
                    # Get legend handles and labels from first axis
                    lines1, labels1 = axs[0].get_legend_handles_labels()

                    all_lines = lines1
                    all_labels = labels1

                    # Create a figure-wide legend
                    fig.legend(all_lines, all_labels, loc='upper left', bbox_to_anchor=(0.95, 0.98))
                
            # Hide the (shared) x-axis
            axs[0].xaxis.set_visible(False)

            
            fig.subplots_adjust(hspace=0)
            # fig.tight_layout(pad=0.0, w_pad=0.0, h_pad=0.0, rect=rect_double)
            fig.savefig(os.path.join(plot_dir, f"{i+1:02}_"+name), dpi=300)
            # fig.tight_layout(pad=0.0, w_pad=0.0, h_pad=0.0)
            # fig.savefig(plot_dir+f"{i+1:02}_"+name, bbox_inches='tight', dpi=300)
            plt.close()

        if plot_seperate_legend:
            fig_leg = plt.figure(figsize=(8., 2./3.)) # if 1 particle, use (8,2) for 3 particles
            ax_leg = fig_leg.add_subplot(111)
            
            # add the legend from the previous axes
            lines1, labels1 = axs[0].get_legend_handles_labels()

            all_lines = lines1
            all_labels = labels1
            entries = len(labels)+1
            
            if ncol is not None:
                ax_leg.legend(all_lines, all_labels, ncol=ncol, loc='center')
            elif entries >= 4:
                ax_leg.legend(all_lines, all_labels, ncol=(entries+1)//2, loc='center')
            else:
                ax_leg.legend(all_lines, all_labels, ncol=entries, loc='center')
            # hide the axes frame and the x/y labels
            ax_leg.axis('off')
            fig_leg.savefig(os.path.join(plot_dir,f"{0:02}_"+"legend.pdf"), bbox_inches='tight', dpi=300, pad_inches=0.1)

            plt.close()

    if not summary_plot:
        return

    # Plot all the histogramms in one file
    number_of_plots = len(plots)
    rows = number_of_plots // plots_per_row
    if number_of_plots%plots_per_row != 0:
        rows += 1
    heights = [1, 0.3, 0.3]*rows

    fig, axs = plt.subplots(rows*3,plots_per_row, dpi=500, figsize=(plots_per_row*7,6*np.sum(heights)), gridspec_kw={'height_ratios': heights})

    iteration = 0
    for i in range(rows*3):
        
        if i%3 == 1:
            iteration -= plots_per_row
            
        for j in range(plots_per_row):
            
            if i % 3 == 2:
                # Add one (small) invisible plot as whitespace
                axs[i,j].set_visible(False)
                continue
            
            elif iteration >= number_of_plots:
                    # Plots are empty remove them
                    axs[i,j].set_visible(False)
                    iteration += 1
                    continue
            
            
            # Select the correct plot input for this axis
            func, name, args1, args2 = plots[iteration]
            
            
            if i % 3 == 0:
                # plot the main data
                plot_hist(
                        file_name=None,
                        data=[func(x, c, **args1) for x, c in zip(xs[1:], cs[1:])],
                        reference=func(xs[0], cs[0], **args1),
                        ax=axs[i,j],
                        panel_ax=axs[i+1,j],
                        labels=labels,
                        errorbars_fake=errorbars_fake,
                        errorbars_true=errorbars_true,
                        y_label=iteration % plots_per_row == 0,
                        fig=fig,
                        print_means=print_means,
                        **args2)
                
                # Hide the (shared) x-axis
                axs[i,j].xaxis.set_visible(False)
                
                
                if i == 0 and j==0:
                    # Get legend handles and labels from first axis
                    lines1, labels1 = axs[0,0].get_legend_handles_labels()

                    # Get legend handles and labels from second axis
                    lines2, labels2 = axs[1,0].get_legend_handles_labels()

                    # Combine handles and labels from both axes
                    all_lines = lines1 + lines2
                    all_labels = labels1 + labels2

                    # Create a figure-wide legend
                    fig.legend(all_lines, all_labels, loc='upper left', bbox_to_anchor=(1,0.5))
                     
                iteration += 1

            if i % 3 == 1:
                iteration += 1             

    fig.subplots_adjust(hspace=0)
    if plot_dir is not None and summary_plot_name is not None:
        fig.savefig(os.path.join(plot_dir,summary_plot_name), bbox_inches='tight', dpi=500)
        plt.close()
    else:
        plt.show()
        

In [ ]:
params = get_plot_params(layer_boundaries_post, coordinates.cpu().numpy(), used_layers=used_layers)
plot_dir = r"C:\Users\flori\OneDrive\Daten\Promotion\Machine Learning\CaloINN\notebooks\plots"

plot_all_hist([x_post.cpu().numpy(), x_post.cpu().numpy()], [c_post.cpu().numpy(), c_post.cpu().numpy()], params, plot_dir=plot_dir, summary_plot=True,
              summary_plot_name="summary.pdf", errorbars_true=True, errorbars_fake=True, ncol=5)